# Conventional Machine Learning
## A Hands-On Workshop for Biomedical Professionals

**Course**: AI in Medicine — Summer Course 2026  
**Date**: 27 July 2026  
**Level**: Beginner to Intermediate  
**Environment**: Google Colab  
**Datasets**: Breast Cancer Wisconsin & Diabetes (scikit-learn)

---
### What We'll Build Today

| # | Section | Duration |
|----|---------|----------|
| 1 | What is Machine Learning? | ~15 min |
| 2 | Classification I — Logistic Regression | ~30 min |
| 3 | Model Evaluation | ~20 min |
| 4 | Classification II — Decision Trees & Random Forest | ~25 min |
| 5 | Regression | ~25 min |
| 6 | Mini Project — Breast Cancer Classification | ~50 min |
| 7 | Wrap-up & Next Steps | ~10 min |

---
### Why Machine Learning for Biomedicine?

- **Diagnosis**: Classify tumors as malignant or benign from cell measurements
- **Prognosis**: Predict disease progression from patient data
- **Screening**: Identify high-risk patients from routine lab results
- **Research**: Discover which biomarkers matter most for outcomes

> Today you'll train models that make real medical predictions — and learn to evaluate whether they're trustworthy.

---
### Before We Start
All libraries are pre-installed in Colab. Just run the imports below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn — the ML workhorse
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.datasets import load_breast_cancer, load_diabetes

sns.set_style('whitegrid')
%matplotlib inline

print('scikit-learn:', __import__('sklearn').__version__)
print('All libraries loaded!')

---
## Section 1: What is Machine Learning?

**Machine Learning** = algorithms that learn patterns from data, then make predictions on new data.

### Two Main Types

| Type | What it does | Biomedical example |
|------|-------------|-------------------|
| **Classification** | Predict a category | Malignant vs benign tumor |
| **Regression** | Predict a number | Disease progression score |

### The scikit-learn Recipe
Almost every ML task in scikit-learn follows the same 4-step pattern:

```python
# 1. Import the model
from sklearn.some_module import SomeModel

# 2. Create an instance
model = SomeModel()

# 3. Train (fit) on data
model.fit(X_train, y_train)

# 4. Predict on new data
predictions = model.predict(X_test)
```

### Train/Test Split — The Golden Rule
**Never evaluate a model on the same data it was trained on.** Always hold out a test set.

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
```

> 🎯 **Key insight**: ML isn't magic. It's pattern recognition at scale. If there's no pattern in your data, no algorithm can find one.

---
## Section 2: Classification I — Logistic Regression

Despite the name, logistic regression is a **classification** algorithm. It estimates the probability that a sample belongs to a class.

**Our dataset**: [Breast Cancer Wisconsin](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) — 569 samples, 30 features from cell nuclei images. Goal: predict malignant vs benign.

### Load & Explore the Data

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target  # 0 = malignant, 1 = benign

print(f'Samples:  {X.shape[0]}')
print(f'Features: {X.shape[1]}')
print(f'Classes:  {cancer.target_names}')
print(f'\nMalignant (0): {np.sum(y == 0)} samples')
print(f'Benign (1):    {np.sum(y == 1)} samples')

In [ ]:
# Convert to DataFrame for easier exploration
df_cancer = pd.DataFrame(X, columns=cancer.feature_names)
df_cancer['target'] = y
df_cancer['diagnosis'] = df_cancer['target'].map({0: 'Malignant', 1: 'Benign'})

print('First 5 samples:')
df_cancer[['diagnosis'] + list(cancer.feature_names[:5])].head()

In [ ]:
# Look at one feature: mean radius
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_cancer[df_cancer['diagnosis'] == 'Malignant']['mean radius'],
             bins=20, alpha=0.7, color='coral', label='Malignant')
axes[0].hist(df_cancer[df_cancer['diagnosis'] == 'Benign']['mean radius'],
             bins=20, alpha=0.7, color='steelblue', label='Benign')
axes[0].set_title('Mean Radius by Diagnosis')
axes[0].set_xlabel('Mean Radius')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot of several features
features_to_plot = ['mean radius', 'mean texture', 'mean perimeter', 'mean area']
df_melted = df_cancer.melt(id_vars=['diagnosis'], value_vars=features_to_plot,
                            var_name='Feature', value_name='Value')
sns.boxplot(data=df_melted, x='Feature', y='Value', hue='diagnosis',
            palette={'Malignant': 'coral', 'Benign': 'steelblue'}, ax=axes[1])
axes[1].set_title('Feature Distributions by Diagnosis')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### Train/Test Split & Scale
Always split BEFORE scaling to avoid data leakage.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')
print(f'Train class balance: {np.mean(y_train):.1%} benign')
print(f'Test  class balance: {np.mean(y_test):.1%} benign')

In [ ]:
# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit ONLY on train
X_test_scaled = scaler.transform(X_test)         # transform test with SAME scaler

print('Scaling complete — all features now have mean≈0, std≈1')

### Train Logistic Regression

In [ ]:
# The 4-step scikit-learn recipe
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = logreg.predict(X_test_scaled)
y_prob = logreg.predict_proba(X_test_scaled)[:, 1]  # probability of benign

# Quick accuracy check
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')
print(f'Correctly classified: {np.sum(y_pred == y_test)} out of {len(y_test)}')

### Confusion Matrix
Accuracy alone can be misleading. The confusion matrix shows where errors happen.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=cancer.target_names))

> 🤔 **Think**: A false negative (predicting benign when it's malignant) is far worse than a false positive in cancer diagnosis. This is why accuracy alone isn't enough — we care about **recall** (sensitivity).

> 🎯 **Classification Takeaway**: Logistic regression is simple, fast, and interpretable. With scaling, it's often competitive with more complex models — always try it first.

---
## Section 3: Model Evaluation

How good is our model? Accuracy tells part of the story. Let's look at the full picture.

### Key Metrics

| Metric | Formula | What it means |
|--------|---------|--------------|
| **Accuracy** | (TP+TN) / Total | How often correct overall |
| **Precision** | TP / (TP+FP) | Of predicted positives, how many are real? |
| **Recall (Sensitivity)** | TP / (TP+FN) | Of real positives, how many did we catch? |
| **F1 Score** | 2×P×R / (P+R) | Harmonic mean of precision & recall |

> In cancer screening: high recall (catch every case) > high precision (avoid false alarms). In drug toxicity screening: the opposite.

In [ ]:
# Calculate all metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}  (sensitivity)')
print(f'F1 Score:  {f1:.3f}')

### ROC Curve & AUC
Shows the tradeoff between true positive rate and false positive rate at every threshold.

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curve — Logistic Regression', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print(f'AUC = {roc_auc:.3f} → {"Excellent" if roc_auc > 0.9 else "Good" if roc_auc > 0.8 else "Fair"} discrimination')

### ✏️ Your Turn (5 min)
Run logistic regression WITHOUT scaling (use `X_train` and `X_test` directly). Compare the accuracy and AUC to the scaled version. Is scaling important for this model?

In [ ]:
# Your experiment here





---
## Section 4: Classification II — Decision Trees & Random Forest

### Decision Tree
A flowchart-like model that splits data by asking yes/no questions about features. Highly interpretable — you can literally draw the tree.

In [ ]:
# Train a decision tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)  # shallow tree = interpretable
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
print(f'Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.3f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_dt, target_names=cancer.target_names))

In [ ]:
# Visualize the tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt, feature_names=cancer.feature_names, class_names=list(cancer.target_names),
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree (max_depth=3) — Breast Cancer Diagnosis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

> 🤔 **Look at the tree**: The first split is on `worst perimeter`. This means that of all 30 features, the model chose this as the single best question to separate malignant from benign. This is **feature importance** in action.

### Random Forest
An ensemble of many decision trees, each trained on a random subset of data. Averages their predictions — more robust, less prone to overfitting.

In [ ]:
# Train a random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

print(f'Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}')
print(classification_report(y_test, y_pred_rf, target_names=cancer.target_names))

In [ ]:
# Feature importance — which features matter most?
importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]  # top 10

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(10), importances[indices], color='steelblue', edgecolor='white')
ax.set_yticks(range(10))
ax.set_yticklabels([cancer.feature_names[i] for i in indices])
ax.set_xlabel('Importance')
ax.set_title('Top 10 Features — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

### Model Comparison

In [ ]:
# Compare all three models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred_m = model.predict(X_test_scaled)
    y_prob_m = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred_m),
        'Precision': precision_score(y_test, y_pred_m),
        'Recall': recall_score(y_test, y_pred_m),
        'F1': f1_score(y_test, y_pred_m),
        'AUC': auc(roc_curve(y_test, y_prob_m)[0], roc_curve(y_test, y_prob_m)[1]) if y_prob_m is not None else np.nan
    })

df_results = pd.DataFrame(results).set_index('Model')
df_results.style.background_gradient(cmap='Blues', axis=0)

> 🎯 **Takeaway**: Random Forest usually outperforms a single decision tree. But logistic regression is often close — and far more interpretable. In medicine, **explainability matters**.

---
## Section 5: Regression

Regression predicts a **continuous number** — not a category.

**Our dataset**: [Diabetes](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) — 442 patients, 10 baseline variables, target = disease progression 1 year later.

### Load & Explore

In [ ]:
# Load diabetes dataset
diabetes = load_diabetes()
X_dia = diabetes.data
y_dia = diabetes.target

df_dia = pd.DataFrame(X_dia, columns=diabetes.feature_names)
df_dia['progression'] = y_dia

print(f'Samples: {X_dia.shape[0]}')
print(f'Features: {X_dia.shape[1]}')
print(f'Target range: {y_dia.min():.0f} to {y_dia.max():.0f}')
print(f'Target mean:  {y_dia.mean():.0f}')
df_dia.describe()

In [ ]:
# Visualize the target
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_dia, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(y_dia.mean(), color='coral', linestyle='--', linewidth=2, label=f'Mean = {y_dia.mean():.0f}')
ax.set_xlabel('Disease Progression (1 year)')
ax.set_ylabel('Number of Patients')
ax.set_title('Diabetes Disease Progression — Target Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### Train Linear Regression

In [ ]:
# Split & scale
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_dia, y_dia, test_size=0.2, random_state=42
)

scaler_dia = StandardScaler()
X_train_d_s = scaler_dia.fit_transform(X_train_d)
X_test_d_s = scaler_dia.transform(X_test_d)

# Train
lr = LinearRegression()
lr.fit(X_train_d_s, y_train_d)
y_pred_d = lr.predict(X_test_d_s)

# Evaluate
print(f'R-squared (R²): {r2_score(y_test_d, y_pred_d):.3f}')
print(f'MAE:            {mean_absolute_error(y_test_d, y_pred_d):.1f}')
print(f'RMSE:           {np.sqrt(mean_squared_error(y_test_d, y_pred_d)):.1f}')
print(f'\nTarget mean: {y_test_d.mean():.0f}, std: {y_test_d.std():.0f}')

### Actual vs Predicted
The gold-standard plot for regression.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test_d, y_pred_d, alpha=0.6, color='steelblue', edgecolors='white')
ax.plot([y_test_d.min(), y_test_d.max()], [y_test_d.min(), y_test_d.max()],
        '--', color='coral', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Progression')
ax.set_ylabel('Predicted Progression')
ax.set_title('Actual vs Predicted — Linear Regression', fontweight='bold')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Residuals
residuals = y_test_d - y_pred_d
print(f'Mean residual (bias): {residuals.mean():.1f} (should be ≈0)')
print(f'Std of residuals:     {residuals.std():.1f}')

### Feature Importance in Linear Regression
The coefficients tell you which features push the prediction up or down.

In [ ]:
# Coefficients (already scaled, so comparable)
coef_df = pd.DataFrame({
    'Feature': diabetes.feature_names,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['coral' if c > 0 else 'steelblue' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Coefficient (effect on progression)')
ax.set_title('Linear Regression Coefficients — Which features matter most?', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top features that INCREASE predicted progression:')
print(coef_df.head(3).to_string(index=False))
print('\nFeatures that DECREASE predicted progression:')
print(coef_df.tail(3).to_string(index=False))

> 🤔 **Clinical interpretation**: BMI and blood pressure (bp) are the strongest predictors of diabetes progression. This aligns with clinical knowledge — which builds trust in the model.

> 🎯 **Regression Takeaway**: R² tells you what fraction of variance your model explains (0 to 1). R² near 1 = great. R² near 0 = your model is no better than guessing the mean. For the diabetes data, R²≈0.45 is typical — disease progression isn't fully predictable from these 10 variables alone.

---
## Section 6: Mini Project — Breast Cancer Classification

Now you'll build a complete ML pipeline — from raw data to model comparison — on the breast cancer dataset.

**Your tasks**:
1. Load and explore the breast cancer data
2. Train at least 3 different models (logistic regression, decision tree, random forest)
3. Compare them using accuracy, precision, recall, F1, and AUC
4. Identify the most important features
5. Answer: Which model would you deploy in a clinical setting? Why?

### Q1: Load & Split
Load the breast cancer dataset, split into train/test (80/20, stratified), and scale the features.

In [ ]:
# Q1: Your code here





In [ ]:
# 💡 Solution
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f'Train: {X_train_s.shape[0]}, Test: {X_test_s.shape[0]}')

### Q2: Train Three Models
Train logistic regression, decision tree (max_depth=5), and random forest (100 trees, max_depth=5). Store predictions.

In [ ]:
# Q2: Your code here





In [ ]:
# 💡 Solution
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

predictions = {}
probabilities = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    predictions[name] = model.predict(X_test_s)
    probabilities[name] = model.predict_proba(X_test_s)[:, 1]
    print(f'{name}: trained')

### Q3: Compare Performance
Create a comparison table with accuracy, precision, recall, F1, and AUC for all three models. Which performs best?

In [ ]:
# Q3: Your code here





In [ ]:
# 💡 Solution
results = []
for name in models:
    y_pred = predictions[name]
    y_prob = probabilities[name]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC': auc(fpr, tpr)
    })

df_compare = pd.DataFrame(results).set_index('Model')
df_compare.style.background_gradient(cmap='Blues', axis=0)

### Q4: ROC Curves
Plot all three ROC curves on the same chart.

In [ ]:
# Q4: Your code here





In [ ]:
# 💡 Solution
fig, ax = plt.subplots(figsize=(8, 7))
colors = {'Logistic Regression': 'steelblue', 'Decision Tree': 'coral', 'Random Forest': 'green'}

for name, model in models.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2.5, color=colors[name], label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout(); plt.show()

### Q5: Feature Importance
Extract and plot the top 10 most important features from the random forest model.

In [ ]:
# Q5: Your code here





In [ ]:
# 💡 Solution
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-10:]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(10), importances[indices], color='steelblue', edgecolor='white')
ax.set_yticks(range(10))
ax.set_yticklabels([cancer.feature_names[i] for i in indices])
ax.set_xlabel('Importance')
ax.set_title('Top 10 Features — Random Forest', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

### Q6: Clinical Decision

**Answer in your own words**: Which model would you deploy in a clinical setting, and why? Consider:
- Accuracy and ROC AUC
- Interpretability (can a doctor understand why the model made a prediction?)
- Speed (how fast does it predict?)
- Which errors are more dangerous — false positives or false negatives?

*Your answer here — double-click to edit*

I would deploy **Logistic Regression** because...



> 🎯 **Mini Project Takeaway**: You just built, evaluated, and compared three ML models on a real biomedical dataset — and made a clinical deployment decision. This is what ML in medicine looks like.

---
## Section 7: Wrap-Up

### What You Learned Today

| Skill | Tool | Biomedical Application |
|-------|------|----------------------|
| Train/test split | `train_test_split` | Fair evaluation of any model |
| Feature scaling | `StandardScaler` | Required for linear models |
| Logistic regression | `LogisticRegression` | Binary diagnosis (malignant/benign) |
| Decision trees | `DecisionTreeClassifier` | Interpretable rules for triage |
| Random forest | `RandomForestClassifier` | Robust, high-accuracy classification |
| Confusion matrix | `confusion_matrix` | Understand where errors happen |
| ROC/AUC | `roc_curve`, `auc` | Model discrimination quality |
| Linear regression | `LinearRegression` | Predict disease progression |
| Model comparison | Multiple metrics | Choose the right model for the job |

### Key scikit-learn Cheat Sheet

```python
# Imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc, r2_score

# Always split first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# Scale (for linear models)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train
model = LogisticRegression()
model.fit(X_train_s, y_train)

# Predict & evaluate
y_pred = model.predict(X_test_s)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
```

### Where to Go From Here

1. **More algorithms** — SVM, k-NN, Gradient Boosting (XGBoost)
2. **Cross-validation** — `cross_val_score` for robust evaluation
3. **Hyperparameter tuning** — `GridSearchCV` to find optimal settings
4. **Deep learning** — `PyTorch` or `TensorFlow` for image/text/sequence data
5. **Deployment** — Turn your model into a web app or API

---

### Thank You!

**You can now build, evaluate, and interpret ML models on real biomedical data.**

Questions? Ask now!